Task:locating 3 ambulances in Stillwater. There are 5 sites under consideration. Ideally, the 3 ambulances should serve all 6 landmarks below within a 3-minute response time. However, this might not be possible, so you decide to maximize the number of them that are covered.

In [ ]:
time_min = { (1,'Airport') : 2, (1,'Boomer Lake') : 3, (1,'Hideaway') : 10,
             (1,'Couch Park') : 15, (1,'Movies') : 5, (1,'Edmon Low') : 7,
             (2,'Airport') : 5, (2,'Boomer Lake') : 6, (2,'Hideaway') : 7,
             (2,'Couch Park') : 12, (2,'Movies') : 14, (2,'Edmon Low') : 8,
             (3,'Airport') : 9, (3,'Boomer Lake') : 8, (3,'Hideaway') : 1,
             (3,'Couch Park') : 4, (3,'Movies') : 8, (3,'Edmon Low') : 5,
             (4,'Airport') : 11, (4,'Boomer Lake') : 10, (4,'Hideaway') : 4,
             (4,'Couch Park') : 3, (4,'Movies') : 9, (4,'Edmon Low') : 8,
             (5,'Airport') : 5, (5,'Boomer Lake') : 3, (5,'Hideaway') : 9,
             (5,'Couch Park') : 10, (5,'Movies') : 3, (5,'Edmon Low') : 9 }

In [49]:
k = 3
sites = { i for (i,j) in time_min.keys() }
landmarks = { j for (i,j) in time_min.keys() }
print(k,sites,landmarks)

3 {1, 2, 3, 4, 5} {'Airport', 'Edmon Low', 'Movies', 'Couch Park', 'Hideaway', 'Boomer Lake'}


In [50]:
close_sites = { j : list() for j in landmarks }
for j in landmarks:
    for i in sites:
        if time_min[i,j] <= 3:
            close_sites[j].append(i)
    print("The sites close (<= 3 min) to",j,"are",close_sites[j])

The sites close (<= 3 min) to Airport are [1]
The sites close (<= 3 min) to Edmon Low are []
The sites close (<= 3 min) to Movies are [5]
The sites close (<= 3 min) to Couch Park are [4]
The sites close (<= 3 min) to Hideaway are [3]
The sites close (<= 3 min) to Boomer Lake are [1, 5]


In [12]:
!pip install --upgrade pip

import gurobipy as gp
from gurobipy import GRB

DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/jupyter-1.0.0-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1


In [51]:
m = gp.Model()

x = m.addVars( landmarks, vtype=GRB.BINARY )
y = m.addVars( sites, vtype=GRB.BINARY )

In [52]:
m.setObjective( gp.quicksum(x) , GRB.MAXIMIZE )

/var/folders/6r/g7p7808s1fschvz6srm5pyxc0000gn/T/ipykernel_12651/1041688146.py:1: DeprecationWarning: Calling quicksum on a tupledict is deprecated, use .sum() instead.
  m.setObjective( gp.quicksum(x) , GRB.MAXIMIZE )


In [53]:
m.addConstr( gp.quicksum(y) == k )

m.addConstrs( gp.quicksum( y[i] for i in close_sites[j] ) >= x[j] for j in landmarks )

/var/folders/6r/g7p7808s1fschvz6srm5pyxc0000gn/T/ipykernel_12651/1474325662.py:1: DeprecationWarning: Calling quicksum on a tupledict is deprecated, use .sum() instead.
  m.addConstr( gp.quicksum(y) == k )


{'Airport': <gurobi.Constr *Awaiting Model Update*>,
 'Edmon Low': <gurobi.Constr *Awaiting Model Update*>,
 'Movies': <gurobi.Constr *Awaiting Model Update*>,
 'Couch Park': <gurobi.Constr *Awaiting Model Update*>,
 'Hideaway': <gurobi.Constr *Awaiting Model Update*>,
 'Boomer Lake': <gurobi.Constr *Awaiting Model Update*>}

In [54]:
m.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 11 columns and 17 nonzeros
Model fingerprint: 0xe8cf6810
Variable types: 0 continuous, 11 integer (11 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+00, 3e+00]
Found heuristic solution: objective 2.0000000
Presolve removed 7 rows and 11 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 4 2 

Optimal solution found (tolerance 1.00e-04)
Best objective 4.000000000000e+00, best bound 4.000000000000e+00, gap 0.0000%


In [55]:
chosen_sites = [ i for i in sites if y[i].x > 0.5 ]
print("Chosen ambulance sites =",chosen_sites)

# Which landmarks do they cover?
covered_landmarks = [ j for j in landmarks if x[j].x > 0.5 ]
print("They cover landmarks =",covered_landmarks)

Chosen ambulance sites = [1, 4, 5]
They cover landmarks = ['Airport', 'Movies', 'Couch Park', 'Boomer Lake']
